# Veg comparisons
---
*J. Michelle Hu  
University of Utah  
March 2025*


In [ ]:
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyproj
import xarray as xr

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

In [ ]:
%reload_ext autoreload
%autoreload 2

### Env setup

In [ ]:
from pathlib import PurePath
# Locate pyproj_datadir for studio env
# From https://stackoverflow.com/questions/69630630/on-fresh-conda-installation-of-pyproj-pyproj-unable-to-set-database-path-pypr
CONDA_ENV = 'studio'
miniconda_dir = '/uufs/chpc.utah.edu/common/home/u6058223/software/pkg/miniconda3'
proj_version = h.fn_list(miniconda_dir, f'envs/{CONDA_ENV}/conda-meta/proj-[0-9]*.json')[0]

VERSION = PurePath(proj_version).stem
pyprojdatadir = f'{miniconda_dir}/pkgs/{VERSION}/share/proj'
print(pyprojdatadir)
pyproj.datadir.set_data_dir(pyprojdatadir)

In [ ]:
def clean_axes(ax, ticksoff=True, labelsoff=True, gridon=True, fc='k', aspect='equal'):
    if ticksoff:
        ax.set_xticks([])
        ax.set_yticks([])
    if labelsoff:
        ax.set_xlabel('')
        ax.set_ylabel('')
    if fc is not None:
        ax.set_facecolor(fc)
    if gridon:
        ax.grid(True)
    ax.set_aspect(aspect)

This notebook compares snow parameters (accumulation: peak timing, peak depth; snowmelt: melt onset, snow disappearance date), energy balance terms (LW, SW, turbulent flux), and albedo (MODIS) with vegetation parameters including: vegetation height, vegetation type, and vegetation density.

We begin by examining basin-specific vegetation characteristics start with the LANDFIRE vegetation files which are used in iSnobal computations. 
- Each basin's topo.nc files contain both vegetation height and vegetation type as well as transmissivity (veg_tau), which we will use as a proxy for density (also based off of veg_type). 
- Additionally, the topo.nc's contain sky view factor, slope, and elevation, which may be of interest.

In [ ]:
# workdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/model_runs/'
# script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'

In [ ]:
# # Basin-specific variables
# # basin = 'blue'
# basin = 'animas'
# # basin = 'yampa'


# # Select just for all variable output of time decay
# basindirs = h.fn_list(workdir, f'*{basin}*/wy*/{basin}*/')
# _ = [print(b) for b in basindirs]
# # Get the WY from the directory name - assumes there is only one WY per basin right now
# WYs = [int(basindir.split('wy')[1][:4]) for basindir in basindirs]
# WYs = np.unique(WYs)

# if len(WYs) > 1:
#     print(f'Multiple water years in {basin} basin: {WYs}')
# elif len(WYs) == 0:
#     print(f'No water years detected in {basin} basin')

# bdx = 1
# WY = WYs[bdx]
# print(f'{WY} selected')

# # Adjust basin dirs based on WY
# basindirs = h.fn_list(workdir, f'{basin}*/wy{WY}/{basin}*/')
# wydir = h.fn_list(workdir, f'{basin}*/wy{WY}/')[0]

# # Figure out filenames
# poly_fn = h.fn_list(script_dir, f'*{basin}*setup/polys/*shp')[0]
# print(poly_fn)

# # Locate basin setup dir
# basin_setupdir = h.fn_list(script_dir, f'*{basin}*setup')[0]
# print(basin_setupdir)

In [ ]:
def prep_basin(basin, WY,
               workdir='/uufs/chpc.utah.edu/common/home/skiles-group3/model_runs/',
               script_dir='/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts',
               verbose=True
               ):
    # Adjust basin dirs based on WY
    basindirs = h.fn_list(workdir, f'{basin}*/wy{WY}/{basin}*/')
    wydir = h.fn_list(workdir, f'{basin}*/wy{WY}/')[0]
    # Figure out filenames
    poly_fn = h.fn_list(script_dir, f'*{basin}*setup/polys/*shp')[0]
    if verbose:
        print(poly_fn)

    # Locate basin setup dir
    basin_setupdir = h.fn_list(script_dir, f'*{basin}*setup')[0]
    if verbose:
        print(basin_setupdir)

    return basindirs, wydir, poly_fn, basin_setupdir

basin = 'animas'
WY = 2021
basindirs, wydir, poly_fn, basin_setupdir = prep_basin(basin=basin, WY=WY)

In [ ]:
import seaborn as sns
sns.set_palette('icefire')

In [ ]:
# # Locate topo.nc file within setupdir
# topo_file = h.fn_list(basin_setupdir, '*/*topo.nc')[0]
# print(topo_file)

In [ ]:
# topo = xr.open_dataset(topo_file)
# topo

In [ ]:
# # Plot the DEM for this basin
# dem = topo['dem']
# dem.plot.imshow()


In [ ]:
def get_basin_terrain(basin_setupdir, verbose=True):
    # Locate topo.nc file within setupdir
    topo_file = h.fn_list(basin_setupdir, '*/*topo.nc')[0]
    if verbose:
        print(topo_file)
    topo = xr.open_dataset(topo_file)
    # topo

    # Extract the DEM for this basin
    dem = topo['dem']
    # dem.plot.imshow()
    return dem

dem = get_basin_terrain(basin_setupdir=basin_setupdir)

## Veg stuff
Potapov GEDI-Landsat global forest height (doesn't look as good, skip this)

In [ ]:
# veg_dir = '/uufs/chpc.utah.edu/common/home/u6058223/veg_data'
savefig = False
outdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures'

In [ ]:
def reproj_match_tcc_to_basin(basin, dem, product='nlcd',
                               veg_dir = '/uufs/chpc.utah.edu/common/home/u6058223/veg_data',
                               verbose=True, EPSG='epsg:32613', ploton=True):
    outname = f'{veg_dir}/{product}_{basin}_reprojmatch.tif'
    if os.path.exists(outname):
        print(f'Output file {outname} already exists. Skipping reprojection, loading directly.')
        return np.squeeze(xr.open_dataset(outname))
    else:
        # NLCD_tcc_conus_2021_v2021-4
        # science_tcc_conus_wgs84_v2023-5_20230101_20231231
        tcc_fn = h.fn_list(veg_dir, f'{product}*{basin}clip.tif')[0]
        if verbose:
            print(tcc_fn)
        tcc = np.squeeze(xr.open_dataset(tcc_fn))
        # reproject to default EPSG (epsg 32613)
        tcc = tcc.rio.reproject(EPSG)

        if ploton:
            fig, axa = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
            ax = axa[0]
            dem.plot.imshow(ax=ax)
            ax = axa[1]
            tcc['band_data'].plot.imshow(ax=ax)
            for ax in axa.flatten():
                ax.set_aspect('equal')
        # Set CRS of DEM
        dem.rio.write_crs(32613, inplace=True)

        # Reproject and match TCC product to the DEM basin
        tcc_reprojmatch = tcc.rio.reproject_match(dem)

        if ploton:
            fig, axa = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
            ax = axa[0]
            dem.plot.imshow(ax=ax)
            ax = axa[1]
            tcc_reprojmatch['band_data'].plot.imshow(ax=ax)
            for ax in axa.flatten():
                ax.set_aspect('equal')

        # Write this out
        tcc_reprojmatch.rio.to_raster(outname)
        return tcc_reprojmatch

In [ ]:
product = 'science'
tcc_reprojmatch = reproj_match_tcc_to_basin(basin=basin, dem=dem, product=product)
tcc_reprojmatch

In [ ]:
# science_tcc_conus_wgs84_v2023-5_20230101_20231231
# do the same with the science data

In [ ]:
landfire = False
if landfire:
     # LANDFIRE
    # Extract vegetation height, vegetation type, transmissivity from the topo.nc

    # Quick plot
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    h.plot_one(topo['veg_height'], specify_ax=(fig, axes[0]), title='Vegetation Height (m)', cmap='Greens', vmin=0, vmax=40)
    h.plot_one(topo['veg_type'], specify_ax=(fig, axes[1]), title='Vegetation Type Code')
    h.plot_one(topo['veg_tau'], specify_ax=(fig, axes[2]), title='Veg Type-Derived Transmissivity', cmap='magma', vmin=0, vmax=1)
    for jdx, ax in enumerate(axes.flatten()):
            clean_axes(ax)
    plt.suptitle(f'{basin.capitalize()} Basin LANDFIRE Veg Height, Type, Tau', y=1.02)
    plt.tight_layout()

    # Save fig
    if savefig:
        fig_fn = f'{outdir}/{basin}_veg_height_type_tau_maps.png'
        fig.savefig(fig_fn, dpi=300, bbox_inches='tight')

    veg_list = [topo['veg_height'], topo['veg_type'], topo['veg_tau']]
    titles = ['Vegetation Height (m)', 'Vegetation Type Code', 'Veg Type-Derived Transmissivity']
    fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
    for jdx, ax in enumerate(axes.flatten()):
        veg_list[jdx].plot.hist(ax=ax, ec='k')
        ax.set_title(titles[jdx])
    plt.suptitle(f'{basin.capitalize()} Basin LANDFIRE Veg Height, Type, Tau', y=1.02)
    plt.tight_layout()
    # Save fig
    if savefig:
        fig_fn = f'{outdir}/{basin}_veg_height_type_tau_hist.png'
        fig.savefig(fig_fn, dpi=300, bbox_inches='tight')

**from 4.3.24 weekly notes:**  
*veg_k*: numpy array of vegetation extinction coefficient from  
            :mod:`smrf.data.loadTopo.Topo`  
*veg_tau*: numpy array of vegetation optical transmissivity from  
            :mod:`smrf.data.loadTopo.Topo`  
  
These values are from Link and Marks (1999), now downloaded in zotero - Distributed simulation of snowcover mass and energy balance  
- Tau_d is optical transmissivity for diffuse radiation of the canopy [dimensionless]  
- K is most likely mu, and is the extinction coefficient [m-1] inversely proportional to optical transmissivity tau_d  

## Snow stuff

In [ ]:
# # Plot check - this helps with troubleshooting unexpected SDD plots, indicates areas of improvement for algorithm selection and algorithm overall
# # Go through solar albedo dir, and plot monthly maps of thickness to cehck on disappearance
# for basindir in basindirs:
#     monthly_ds = xr.open_mfdataset(f'{basindir}/run*01/snow.nc', drop_variables=['snow_density', 'specific_mass', 'liquid_water', 'temp_surf', 'temp_lower',
#                                                                                  'temp_snowcover', 'thickness_lower', 'water_saturation', 'projection'])

#     fig, axes = plt.subplots(3, 4, figsize=(12, 8))
#     vmin = 0
#     vmax = 1
#     for jdx, timeslice in enumerate(monthly_ds['thickness'].isel(time=slice(0, 12))):
#         ax = axes.flatten()[jdx]
#         if jdx < 11:
#             scale = None
#         else:
#             scale = 1
#         h.plot_one(timeslice, specify_ax=(fig, ax),
#                 title=f'{pd.to_datetime(timeslice.time.values).strftime('%Y-%m-%d')}',
#                 vmin=vmin, vmax=vmax, cmap='viridis', scale=scale)
#         clean_axes(ax)
#         ax.set_aspect('equal')

#     model_abbrev = PurePath(basindir).stem.split("_100m")[1]
#     if len(model_abbrev) == 0:
#         model_abbrev = 'Baseline'
#     else:
#         model_abbrev = 'HRRR-MODIS'
#     plt.suptitle(f'{basin.capitalize()} Basin {WY} {model_abbrev} Monthly Snow Depth', y=1.02)
#     plt.tight_layout()

#     # Save fig
#     fig_fn = f'{outdir}/{basin}_wy{WY}_monthly{PurePath(basindir).stem.split("_100m")[1]}_thickness.png'
#     print(fig_fn)

#     if savefig:
#         fig.savefig(fig_fn, dpi=300, bbox_inches='tight')

In [ ]:
# # Some exploratory analysis
# vtype = 'sdd' #'peak'
# snowprop = 'depth'
# var = 'sdd_doy' #'peak_doy'
# abbrev_lab='SDD DOY'

# # Get the calculated files
# # Calculate the timing of peak snow depth for each pixel and plot (histogram, map) - read from file! use calc_basin_peak.py to generate
# calc_fns = h.fn_list(wydir, f'*{vtype}*{snowprop}*.nc')

# # Read them in
# ds_list = [xr.open_dataset(calc_fn) for calc_fn in calc_fns]

# # Calculate the difference (HRRR-MODIS - Baseline)
# diff = ds_list[1][var] - ds_list[0][var]

# difflim = 30
# diffcmap = 'magma'
# vmin, vmax = 0, 180

In [ ]:
# # Plot them
# numrows, numcols = 1, 3
# width, height = 4*numcols, 4*numrows
# sharex, sharey, turnoffaxes, turnofflabels = True, True, True, True
# maskval = 359
# fig, axes = plt.subplots(numrows, numcols, figsize=(width, height), sharex=sharex, sharey=sharey)
# for jdx, ds in enumerate(ds_list):
#     ax = axes[jdx]
#     h.plot_one(np.ma.masked_greater_equal(ds[var], maskval), specify_ax=(fig, ax), vmin=vmin, vmax=vmax, turnoffaxes=turnoffaxes, turnofflabels=turnofflabels,
#                title=f'{PurePath(calc_fns[jdx]).stem.split(f"_{vtype}_{snowprop}")[0]}')
#     ax.set_aspect('equal')
# ax = axes[jdx+1]

# # Plot the difference
# h.plot_one(np.ma.masked_greater_equal(diff, maskval), cmap=diffcmap, specify_ax=(fig, ax),
#            vmin=-difflim, vmax=difflim,
#            turnofflabels=turnofflabels, title=f'HRRR-SPIReS - Baseline {abbrev_lab}')
# plt.suptitle(f'{basin.capitalize()} Basin WY {WY} {abbrev_lab} Maps', y=1.02)
# plt.tight_layout()

# # Save fig
# fig_fn = f'{outdir}/{basin}_wy{WY}_{vtype}_{snowprop}_map.png'
# print(fig_fn)
# if savefig:
#     fig.savefig(fig_fn, dpi=300, bbox_inches='tight')

In [ ]:
# # Plot histograms
# fig, axes = plt.subplots(numrows, numcols, figsize=(12, 3), sharey=sharey)
# for jdx, ds in enumerate(ds_list):
#     ax = axes[jdx]
#     h.plot_hist(ds[var], specify_ax=(fig, ax), title=f'{PurePath(calc_fns[jdx]).stem.split(f"_{vtype}_{snowprop}")[0]}', xlabel=abbrev_lab)
# ax = axes[jdx+1]
# h.plot_hist(diff, specify_ax=(fig, ax), title=f'HRRR-MODIS - Baseline {abbrev_lab}', xlabel=f'{abbrev_lab} diff')
# plt.tight_layout()
# plt.suptitle(f'{basin.capitalize()} Basin WY {WY} {abbrev_lab} Histograms', y=1.02)

# # Save fig
# fig_fn = f'{outdir}/{basin}_wy{WY}_{vtype}_{snowprop}_hist.png'
# print(fig_fn)
# if savefig:
#     fig.savefig(fig_fn, dpi=300, bbox_inches='tight')

In [ ]:
nlcd = False
science_tcc = True

In [ ]:
# for mdx, ds in enumerate(ds_list):
#     # Plot veg height against snow var
#     if landfire:
#         xvar = [topo['veg_height'], topo['veg_type'], topo['veg_tau']]
#         xlab = ['Veg Height (m)', 'Veg Type', 'Veg Transmissivity']
#     elif nlcd:
#         xvar = [nlcd_reprojmatch['band_data']]
#         xlab = ['Veg Height (m)']
#     elif science_tcc:
#         pass
#         # xvar = [tcc['band_data']]
#         # xlab = ['Veg Height (m)']
#     fig, axes = plt.subplots(1, len(xvar), figsize=(6*len(xvar), 4))
#     yvar = ds[var]
#     ylab = 'Snow Disappearance DOY'
#     titles = [f'{ylab} vs. {x}' for x in xlab]
#     if len(xvar) == 1:
#         axes = np.array([axes])
#     for jdx, ax in enumerate(axes.flatten()):
#         ax.scatter(xvar[jdx].values.flatten(), yvar.values.flatten(), color='k', s=20, alpha=0.2)
#         ax.set_xlabel(xlab[jdx])
#         ax.set_ylabel(ylab)
#         ax.set_title(titles[jdx])
#         # ax.set_ylim(0, 120)

#     plt.suptitle(PurePath(calc_fns[mdx]).stem, fontsize=14, y=1.02)

In [ ]:
# x = np.ravel(xvar[0].data)
# jointcolor = 'seagreen'
# jointheight = 4

# for ds in ds_list:
#     yvar = ds[var]
#     y = np.ravel(yvar.data)
#     s = sns.jointplot(x=x, y=y, color=jointcolor,
#                     kind="hist", height=jointheight)

#     # JointGrid has a convenience function
#     s.set_axis_labels('Veg height', 'SDD (DOY)', fontsize=16, fontweight='bold')

#     ax = s.ax_joint
#     ax.set_ylim(0, 200)
#     q = sns.regplot(ax=ax, x=x, y=y, color=jointcolor, scatter=False)
#     # ax.legend()

In [ ]:
def extract_basin_sdd(wydir, vtype='sdd', snowprop='depth'):
    # # Some exploratory analysis
    # vtype = 'sdd' #'peak'
    # snowprop = 'depth'
    # var = 'sdd_doy' #'peak_doy'
    # abbrev_lab='SDD DOY'

    # Get the calculated files
    # Calculate the timing of peak snow depth for each pixel and plot (histogram, map) - read from file! use calc_basin_peak.py to generate
    calc_fns = h.fn_list(wydir, f'*{vtype}*{snowprop}*.nc')

    # Read them in
    ds_list = [xr.open_dataset(calc_fn) for calc_fn in calc_fns]
    return ds_list

ds_list = extract_basin_sdd(wydir=wydir)

In [ ]:
product

In [ ]:
def plot_sdd_tcc_hexbin(basin, WY, xvar, ds_list, product, var='sdd_doy', outdir=None,
                              jointcolor='seagreen', jointheight=4,
                              kind='hex', runtypes=['Baseline', 'HRRR-SPIReS'],
                              marginal_kws = dict(bins=12, fill=False)):
    x = np.ravel(xvar.data)
    for mdx, ds in enumerate(ds_list):
        yvar = ds[var]
        y = np.ravel(yvar.data)

        s = sns.jointplot(x=x, y=y, color=jointcolor,
                          kind=kind, height=jointheight,
                          gridsize=25,
                          )

        # JointGrid has a convenience function
        s.set_axis_labels(f'{basin.capitalize()} % Canopy Cover', f'{runtypes[mdx]} SDD (WY {WY})', fontsize=12, fontweight='bold')

        ax = s.ax_joint
        # Calculate correlation coefficient
        corr = np.corrcoef(x, y)[0, 1]
        print(f'Correlation coefficient: {corr:.2f}')

        # Add to plot
        ax.text(0.05, 0.95, f'Correlation: {corr:.2f}', transform=ax.transAxes,
                fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 200)
        sns.regplot(ax=ax, x=x, y=y, color=jointcolor, scatter=False)
        if outdir is not None:
            outname = f'{outdir}/{basin}_{product}_tcc_{runtypes[mdx]}_sdd_wy{WY}_hexbin.png'
            print(f'Saving figure to {outname}')
            plt.savefig(outname, dpi=300, bbox_inches='tight')

# for NCLD
plot_sdd_tcc_hexbin(basin=basin, WY=WY, xvar=tcc_reprojmatch['band_data'], product=product, ds_list=ds_list, outdir=outdir)


In [ ]:
# Crank out for all the basins and water years
for basin in ['animas', 'blue', 'yampa']:
    print(basin.upper())
    for WY in 2021, 2022, 2023, 2024:
        print(f'Processing WY {WY}')
        basindirs, wydir, poly_fn, basin_setupdir = prep_basin(basin=basin, WY=WY, verbose=False)
        ds_list = extract_basin_sdd(wydir=wydir)
        dem = get_basin_terrain(basin_setupdir=basin_setupdir, verbose=False)

        # For NLCD
        tcc_reprojmatch = reproj_match_tcc_to_basin(basin=basin, dem=dem, ploton=False)

        # Plot hexbin and save the fig
        plot_sdd_tcc_hexbin(basin=basin, WY=WY,
                                  xvar=tcc_reprojmatch['band_data'],
                                  product='science',
                                  ds_list=ds_list,
                                  outdir=outdir)

In [ ]:
kind = 'hex'
runtypes = ['Baseline', 'HRRR-SPIReS']
marginal_kws = dict(bins=12, fill=False)
for mdx, ds in enumerate(ds_list):
    yvar = ds[var]
    y = np.ravel(yvar.data)

    s = sns.jointplot(x=x, y=y, color=jointcolor,
                kind=kind, height=jointheight,
                gridsize=25,
                marginal_kws=marginal_kws
                )
    # JointGrid has a convenience function
    s.set_axis_labels(f'{basin.capitalize()} Veg height', f'{runtypes[mdx]} SDD (DOY)', fontsize=12, fontweight='bold')

    ax = s.ax_joint
    # Calculate correlation coefficient
    corr = np.corrcoef(x, y)[0, 1]
    print(f'Correlation coefficient: {corr:.2f}')

    # Add to plot
    ax.text(0.05, 0.95, f'Correlation: {corr:.2f}', transform=ax.transAxes,
            fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
    ax.set_xlim(0, 60)
    ax.set_ylim(0, 200)
    q = sns.regplot(ax=ax, x=x, y=y, color=jointcolor, scatter=False)


In [ ]:
kind = 'hex'
runtypes = ['Baseline', 'HRRR-SPIReS']
for mdx, ds in enumerate(ds_list):
    yvar = ds[var]
    y = np.ravel(yvar.data)

    s = sns.jointplot(x=x, y=y, color=jointcolor,
                kind=kind, height=jointheight,
                gridsize=25,
                )
    
    # JointGrid has a convenience function
    s.set_axis_labels(f'{basin.capitalize()} Veg height', f'{runtypes[mdx]} SDD (DOY)', fontsize=12, fontweight='bold')

    ax = s.ax_joint
    # Calculate correlation coefficient
    corr = np.corrcoef(x, y)[0, 1]
    print(f'Correlation coefficient: {corr:.2f}')

    # Add to plot
    ax.text(0.05, 0.95, f'Correlation: {corr:.2f}', transform=ax.transAxes,
            fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
    ax.set_xlim(0, 60)
    ax.set_ylim(0, 200)
    q = sns.regplot(ax=ax, x=x, y=y, color=jointcolor, scatter=False)


In [ ]:
sns.set_palette('tab20')

In [ ]:
figsize = (12, 4)
histtype = 'step'
histbins=52
histlw = 1.5
annotcolor = 'k'
annotsize = 12
annotcenterx = 250
suptitlesize = 14
suptitley = 1.02
veglist = [topo['veg_height'], topo['veg_type'], topo['veg_tau']]
palette = 'tab20'

for mdx, ds in enumerate(ds_list):
    xvar = veglist
    yvar = ds[var]
    xlab = ['Veg Height (m)', 'Veg Type', 'Veg Transmissivity']
    ylab = 'Snow Disappearance DOY'
    titles = [f'{ylab} vs. Veg Height', f'{ylab} vs. Veg Type', f'{ylab} vs. Veg Transmissivity']
    for zdx, xv in enumerate(xvar):
        # Skip veg type, there are too many types
        if zdx == 1:
            pass
        else:
            unique_vals = np.unique(xvar[zdx])
            fig, ax = plt.subplots(figsize=figsize)
            for kdx, c in enumerate(unique_vals):
                this_class = yvar.where(xvar[zdx] == c).values
                this_class = this_class[~np.isnan(this_class)]
                ax.hist(this_class, histtype=histtype, bins=histbins, linewidth=histlw, label=f'{c:.2f}')
            ymean = ax.get_ylim()[1] / 3 * 2
            # Annotate the median doy for each class c in the color of the histogram
            classcolors = sns.color_palette(palette, len(unique_vals))
            _ = [ax.annotate(f'{c:.2f}: {np.nanmedian(yvar.where(xvar[zdx] == c).values)}', xy=(annotcenterx, ymean-kdx*ymean*0.1),
                             c=classcolors[kdx], fontsize=annotsize, ha='center') for kdx, c in enumerate(unique_vals)]
            # add the "legend" for annotations
            ax.annotate(f'{xlab[zdx]}: {abbrev_lab}', xy=(annotcenterx, ymean+ymean*0.1), c=annotcolor, fontsize=annotsize, ha='center')
            plt.legend()
            plt.title(f'{ylab} Histogram by {xlab[zdx]}')
            plt.suptitle(PurePath(calc_fns[mdx]).stem, fontsize=suptitlesize, y=suptitley)

            # Save fig
            fig_fn = f'{outdir}/{basin}_wy{WY}_{vtype}_{snowprop}_veg_{xlab[zdx].split("Veg ")[1].split(" ")[0].lower()}{PurePath(calc_fns[mdx]).stem.split("_100m")[1].split(f"{vtype}_")[0]}hist.png'
            print(fig_fn)
            if savefig:
                fig.savefig(fig_fn, dpi=300, bbox_inches='tight')

In [ ]:
for mdx, ds in enumerate(ds_list):
    # Now plot it as a heat map
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    xvar = veglist
    yvar = ds[var]
    for jdx, ax in enumerate(axes.flatten()):
        sns.histplot(x=xvar[jdx].values.flatten(), y=yvar.values.flatten(), ax=ax)
        ax.set_xlabel(xlab[jdx])
        ax.set_ylabel(ylab)
        ax.set_title(titles[jdx])
    plt.suptitle(PurePath(calc_fns[mdx]).stem, fontsize=suptitlesize, y=suptitley)

    # Save fig
    fig_fn = f'{outdir}/{basin}_wy{WY}_{vtype}_{snowprop}_veg{PurePath(calc_fns[mdx]).stem.split("_100m")[1].split(f"{vtype}_")[0]}heat.png'
    print(fig_fn)
    if savefig:
        fig.savefig(fig_fn, dpi=300, bbox_inches='tight')

## Notes
**Time decay trends**
- SDD is earliest in the middle veg height classes
- SDD appears latest in the lowest (open) and highest classes (tall trees)
- SDD is earlier with middling veg_tau classes (0.3, 0.44) and latest in the smallest (0.16, most dense) and largest (1.00, least dense) classes

What's going on? Probably elevation interactions with vegetation  
At lower/mid elevations, there is a higher concentration of taller vegetation (high veg height and most dense) where snow may be shielded from solar radiation  
At higher elevations, there is a higher proportion of short veg (low or zero height) and open spaces (high transmissivity). But snow lasts longer at these higher elevations  

**HRRR-MODIS trends**
- SDD is generally earlier with increasing vegetation height, but this is kind of bouncy
- SDD appears latest in the lowest (open) classes. SDD is much earlier in the higher classes (taller trees)
- SDD is earliest with the smallest (0.16, most dense) classes, and latest with the largest (1.00, least dense) classes

This is very different from the baseline (time decay). 


In [ ]:
# figsize = (12, 4)
# histtype = 'step'
# histbins=52
# annotcolor = 'k'
# annotsize = 12
# annotcenterx = 250
# suptitlesize = 14
# suptitley = 1.02
veglist = [topo['veg_height'], topo['veg_type'], topo['veg_tau']]
# palette = 'tab20'
histlw = 3

In [ ]:
annotsize = 16

In [ ]:
# sns.set_palette('colorblind')
palette = 'tab20'

In [ ]:
veg_height_classes = [0.00, 0.25, 0.75, 1.00, 2.00, 2.50, 3.00, 7.50, 17.50, 37.50]
color_dict = {c: sns.color_palette(palette, len(veg_height_classes))[len(veg_height_classes)-kdx-1] for kdx, c in enumerate(veg_height_classes)}
sns.color_palette(palette, len(veg_height_classes))

In [ ]:
color_dict

In [ ]:
color_dict = {c: sns.color_palette(palette, len(veg_height_classes))[kdx] for kdx, c in enumerate(veg_height_classes)}
color_dict

In [ ]:
# Calculate median shift per class
xvar = veglist
yvar = diff
diffannotcenterx = 75
titles = [f'{ylab} shift vs. Veg Height', f'{ylab} shift vs. Veg Type', f'{ylab} shift vs. Veg Transmissivity']
# xlims = (-150, 150)
xlims = (-100, 100)
# Desired bin width
bin_width = 5
for zdx, xv in enumerate(xvar):
    if zdx == 1:
        pass
    else:
        unique_vals = np.unique(xvar[zdx])
        fig, ax = plt.subplots(figsize=(6,4))
        # Set a standard bin width despite variability and range in class values
        for kdx, c in enumerate(unique_vals):
            this_class = yvar.where(xvar[zdx] == c).values
            this_class = this_class[~np.isnan(this_class)]
            # Calculate bin edges
            min_val = np.min(this_class)
            max_val = np.max(this_class)
            bins = np.arange(min_val, max_val + bin_width, bin_width)
            # ax.hist(this_class, histtype=histtype, bins=histbins * 2, linewidth=histlw, label=f'{c:.2f}')
            ax.hist(this_class, histtype=histtype, bins=bins, linewidth=histlw, label=f'{c:.2f}')
        # Annotate the median doy for each class c in the color of the histogram
        classcolors = sns.color_palette(palette, len(unique_vals))
        ymean = ax.get_ylim()[1] / 3 * 2.5
        print(ymean)
        # _ = [ax.annotate(f'{c:.2f}: {np.nanmedian(yvar.where(xvar[zdx] == c).values)}', xy=(diffannotcenterx, ymean-kdx*ymean*0.1),
        #                     c=classcolors[kdx], fontsize=annotsize, ha='center') for kdx, c in enumerate(unique_vals)]
        # # add the "legend" for annotations
        # ax.annotate(f'{xlab[zdx]}: {abbrev_lab}', xy=(diffannotcenterx, ymean+ymean*0.1), c=annotcolor, fontsize=annotsize, ha='center')
        # Add dashed zero line
        ax.axvline(0, linestyle='--', color='k', alpha=0.3)
        ax.set_xlim(xlims)
        # ax.annotate(f'{basin.capitalize()} WY {WY}', xy=(-145, ymean*1.1), c=annotcolor, fontweight='bold', fontsize=14, ha='left')

        # # Plot annotation showing earlier melt vs. later melt than Baseline
        # ax.annotate('HRRR-SPIReS \nmelts earlier', xy=(-100, ymean/5), c='goldenrod', fontsize=annotsize, ha='center')
        # ax.annotate('HRRR-SPIReS \nmelts later', xy=(100, ymean/5), c='b', fontsize=annotsize, ha='center')
        # Save fig
        fig_fn = f'{outdir}/{basin}_wy{WY}_{vtype}_{snowprop}_veg_{xlab[zdx].split("Veg ")[1].split(" ")[0].lower()}_hist_shift.png'
        print(fig_fn)
        if savefig:
            fig.savefig(fig_fn, dpi=300, bbox_inches='tight')

- The shift shows that SDD is later in HRRR-MODIS in the lowest vegetation height classes and much earlier in the taller veg height classes
- Similarly, SDD is much earlier for the lower veg_tau classes (denser veg) and later for the higher veg_tau classes (more open)

**This is counterintuitive. we would expect that more open areas with greater insoluation would melt snow faster, pushing SDD earlier, not later!!**

In [ ]:
np.ravel(topo['veg_tau'])

In [ ]:
np.ravel(topo['veg_tau'])

In [ ]:
np.ravel(topo['veg_tau'])

## Pick up here

In [ ]:
# Calculate the melt onset date for each pixel and plot (histogram, map)
# consistent SWI > 0? - don't have this yet, would want to make a separate script for this
# consistent declining SWE? - see what the swe calculation is like from notes - computed in a notebook, can you pull this into a script?


#### Based on these histograms, the following variables may be of interest to examine:
- 
- 
- 

#### Compare each of the identified variables against veg_height, veg_type, and veg_tau (transmissivity)

In [ ]:
# Do this with scatter plots and correlation coefficients

In [ ]:
# Plot the rose plots for each variable

## Albedo

In [ ]:
albedo_dir = "/uufs/chpc.utah.edu/common/home/skiles-group3/MODIS_albedo"
# Read in the albedo data
albedo_fns = h.fn_list(albedo_dir, f'wy{WY}/*{basin}*24.nc')
print(len(albedo_fns))
basin_albedo = xr.open_mfdataset(albedo_fns)
basin_albedo

In [ ]:
# plot one timestep
basin_albedo['albedo'].isel(time=0).plot()


## Energy